# Сжатие боксов готового сабмита

Каждый бокс сжимается к своему центру в K раз, уверенность не меняется. Модели не нужны — работает прямо по CSV.

Зачем: train размечал один радиолог, а тест соревнования — пересечение разметки трёх, поэтому тестовые боксы меньше. K подбирался по Kaggle, итоги — в README.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [13]:
import pandas as pd
from pathlib import Path
project = Path("/content/drive/MyDrive/rsna_project_final")
K = 0.8
csv = Path("submission_classifier_ConvNeXt_V2_Tiny_no_aug_YOLO26m.csv")
sub = pd.read_csv(project / csv)

def shrink(s):
    if not isinstance(s, str) or not s.strip():
        return ""
    v = list(map(float, s.split()))
    out = []
    for i in range(0, len(v), 5):
        c, x, y, w, h = v[i:i + 5]
        out.append(f"{c:.3f} {x + w * (1 - K) / 2:.1f} {y + h * (1 - K) / 2:.1f} {w * K:.1f} {h * K:.1f}")
    return " ".join(out)

sub["PredictionString"] = sub["PredictionString"].apply(shrink)
sub.to_csv(project / f"{csv.stem}_shrink_{K}.csv", index=False)